# 5 Beginner Quant Projects — Sri Lanka (CSE) Edition — Google Colab

Same 5 projects, adapted for Colombo Stock Exchange (CSE) stocks: COMB, HNB, SAMP, NDB, NTB, DFCC, CTC, JKH, etc.

**Important:** `yfinance` does NOT carry CSE data — Yahoo Finance simply doesn't cover the Colombo Stock Exchange. So instead of auto-downloading, this notebook uses CSV files you export yourself from CSE's historical data tool, then upload into Colab.

### How to get the CSV files (do this before running)
1. Go to **cse.lk**
2. Search for your stock (e.g. "JKH.N0000" or "Commercial Bank")
3. Open the stock's page → find **Historical Data** / **Trade Summary** download
4. Set your date range (e.g. 2020-01-01 to today) and export as CSV
5. Repeat for each stock you want to analyze
6. Keep the downloaded files handy — you'll upload them directly into Colab in Step 1.1 below

If cse.lk's export format changes or a column name doesn't match, Step 1.1 prints your file's actual columns so you can adjust — don't just run blindly, check the printout.


## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from google.colab import files

plt.style.use('seaborn-v0_8-darkgrid') if 'seaborn-v0_8-darkgrid' in plt.style.available else None
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


---
# Project 1: Stock Data Loader & Moving Average Visualizer

**Step 1.1 — Upload your CSE CSV (e.g. JKH.N0000 historical data).**

Click "Choose Files" when prompted and select the CSV you downloaded from cse.lk.


In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

raw = pd.read_csv(filename)
print("Columns found in your file:")
print(raw.columns.tolist())
raw.head()


**Step 1.2 — Standardize the file into Date + Close columns.**

CSE exports commonly use column names like `Trade Date`, `Date`, `Close (Rs.)`, `Close Price`, or `Last Traded Price`. Check the printout above and set the two variable names below to match YOUR file exactly before running this cell.


In [ ]:
DATE_COL = "Trade Date"      # <-- change to match your file's column name
CLOSE_COL = "Close (Rs.)"    # <-- change to match your file's column name

df = raw[[DATE_COL, CLOSE_COL]].copy()
df.columns = ['Date', 'Close']
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Close'] = pd.to_numeric(df['Close'].astype(str).str.replace(',', ''), errors='coerce')
df = df.dropna().sort_values('Date').set_index('Date')

TICKER = filename.split('.')[0]
df.head()


**Step 1.3 — Compute 20-day and 50-day Simple Moving Averages.**

In [ ]:
df['SMA20'] = df['Close'].rolling(20).mean()
df['SMA50'] = df['Close'].rolling(50).mean()
df.tail()


**Step 1.4 — Plot price with both SMAs.**

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df.index, df['Close'], label='Close (LKR)', linewidth=1)
plt.plot(df.index, df['SMA20'], label='SMA 20', linewidth=1.2)
plt.plot(df.index, df['SMA50'], label='SMA 50', linewidth=1.2)
plt.title(f'{TICKER} Price with 20/50-day SMAs (CSE)')
plt.xlabel('Date'); plt.ylabel('Price (LKR)')
plt.legend()
plt.show()


**Step 1.5 — Build the crossover signal.**

In [ ]:
df['Signal'] = np.where(df['SMA20'] > df['SMA50'], 1, 0)
df[['Close','SMA20','SMA50','Signal']].tail(10)


**Beginner checkpoint — CSE-specific note:** CSE trading days differ from US markets (different holidays, and historically lower liquidity for many counters outside the top ~20 stocks). Thin trading can create flat stretches or gaps in your Close series — check for suspiciously long runs of an identical price, which usually means no trades occurred that day rather than genuine price stability.

---
# Project 2: SMA Crossover Backtester


In [ ]:
df['Return'] = df['Close'].pct_change()
df['Strategy_Return'] = df['Signal'].shift(1) * df['Return']
df[['Return','Signal','Strategy_Return']].tail()


In [ ]:
df['Cum_BuyHold'] = (1 + df['Return']).cumprod()
df['Cum_Strategy'] = (1 + df['Strategy_Return']).cumprod()

plt.figure(figsize=(12,6))
plt.plot(df.index, df['Cum_BuyHold'], label='Buy & Hold')
plt.plot(df.index, df['Cum_Strategy'], label='SMA Crossover Strategy')
plt.title(f'{TICKER}: Strategy vs Buy & Hold (growth of LKR 1)')
plt.xlabel('Date'); plt.ylabel('Growth of LKR 1')
plt.legend()
plt.show()


**Step — Sharpe ratio.**

CSE has ~240-ish trading days/year historically (fewer than the US's 252 due to local holidays) — adjust `periods_per_year` if you want a more precise annualization for CSE specifically.


In [ ]:
def sharpe_ratio(returns, periods_per_year=240, risk_free=0.0):
    excess = returns.dropna() - risk_free/periods_per_year
    return (excess.mean() / excess.std()) * np.sqrt(periods_per_year)

print(f"Strategy Sharpe:  {sharpe_ratio(df['Strategy_Return']):.3f}")
print(f"Buy&Hold Sharpe:  {sharpe_ratio(df['Return']):.3f}")


**Beginner checkpoint — CSE-specific note:** Sri Lanka's risk-free rate is meaningfully higher than the US's (CBSL Treasury bill/bond yields, not the US Fed funds rate) — if you want a proper Sharpe ratio rather than the simplified zero-risk-free version above, plug in the current LKR T-bill yield as `risk_free`, not a US rate.

---
# Project 3: Black-Scholes Option Pricer

**Note:** CSE does not have a listed, liquid equity options market like the US — this project is still valuable as a modeling exercise (and directly useful if you ever price OTC/structured products), but treat it as educational rather than something you'd apply to a live CSE options book.


In [ ]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if option_type == 'call':
        price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else:
        price = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    return price, d1, d2

# Example using a CSE stock's current price as S, an LKR risk-free rate as r
S = df['Close'].iloc[-1]      # latest price from your uploaded stock
K = round(S / 5) * 5          # a nearby round-number strike
T = 1.0
r = 0.09                      # example: ~9% LKR T-bill yield -- update to current CBSL rate
sigma = df['Return'].std() * np.sqrt(240)   # historical annualized volatility from YOUR data

call_price, d1, d2 = black_scholes(S, K, T, r, sigma, 'call')
put_price, _, _ = black_scholes(S, K, T, r, sigma, 'put')

print(f"Spot (S): {S:.2f} LKR | Strike (K): {K:.2f} LKR | Annualized vol: {sigma:.2%}")
print(f"Call price: {call_price:.4f} LKR")
print(f"Put price:  {put_price:.4f} LKR")


**Step — put-call parity validation and Greeks (unchanged logic, LKR-denominated).**

In [ ]:
lhs = call_price - put_price
rhs = S - K*np.exp(-r*T)
print(f"Call - Put = {lhs:.4f} | S - K*e^-rT = {rhs:.4f} | Match: {np.isclose(lhs, rhs)}")

def greeks(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))
    vega  = S*norm.pdf(d1)*np.sqrt(T) / 100
    if option_type == 'call':
        delta = norm.cdf(d1)
        theta = (-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)) / 365
    else:
        delta = norm.cdf(d1) - 1
        theta = (-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) + r*K*np.exp(-r*T)*norm.cdf(-d2)) / 365
    return {'Delta': delta, 'Gamma': gamma, 'Vega': vega, 'Theta (per day)': theta}

print("Call Greeks:", greeks(S, K, T, r, sigma, 'call'))


**Beginner checkpoint — CSE-specific note:** Notice `sigma` here comes from YOUR actual historical CSE stock returns, not an assumed number — this matters a lot in Sri Lanka's context, since post-2022-default volatility regimes are very different from pre-crisis ones (something your dissertation's regime-instability findings already demonstrate directly).

---
# Project 4: Markowitz Portfolio Optimization (Efficient Frontier) — CSE Banking Basket

**Step 4.1 — Upload multiple CSE CSVs (one per stock).**

Suggested basket, matching your dissertation panel: COMB, HNB, SAMP, NDB, NTB, DFCC.


In [ ]:
print("Upload one CSV per stock (you can select multiple files at once).")
uploaded_multi = files.upload()

def load_cse_csv(filename, date_col, close_col):
    raw = pd.read_csv(filename)
    d = raw[[date_col, close_col]].copy()
    d.columns = ['Date', 'Close']
    d['Date'] = pd.to_datetime(d['Date'], dayfirst=True, errors='coerce')
    d['Close'] = pd.to_numeric(d['Close'].astype(str).str.replace(',', ''), errors='coerce')
    return d.dropna().sort_values('Date').set_index('Date')['Close']

# IMPORTANT: check each file's real column names first (like Step 1.1) and adjust below
DATE_COL = "Trade Date"
CLOSE_COL = "Close (Rs.)"

price_dict = {}
for fname in uploaded_multi.keys():
    ticker_name = fname.split('.')[0]
    price_dict[ticker_name] = load_cse_csv(fname, DATE_COL, CLOSE_COL)

prices = pd.DataFrame(price_dict).dropna()
TICKERS = prices.columns.tolist()
prices.head()


**Step 4.2 — Annualized mean returns and covariance matrix.**

In [ ]:
returns = prices.pct_change().dropna()
mean_returns_annual = returns.mean() * 240
cov_matrix_annual = returns.cov() * 240

print("Annualized mean returns:\n", mean_returns_annual)
print("\nAnnualized covariance matrix:\n", cov_matrix_annual)


**Step 4.3 — Simulate random portfolios.**

In [ ]:
np.random.seed(42)
n_portfolios = 5000
n_assets = len(TICKERS)

results = np.zeros((3, n_portfolios))
weights_record = []

for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= np.sum(w)
    weights_record.append(w)
    port_return = np.dot(w, mean_returns_annual)
    port_vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix_annual, w)))
    results[0, i] = port_return
    results[1, i] = port_vol
    results[2, i] = port_return / port_vol


**Step 4.4 — Plot the efficient frontier.**

In [ ]:
max_sharpe_idx = np.argmax(results[2])
min_vol_idx = np.argmin(results[1])

plt.figure(figsize=(12,7))
sc = plt.scatter(results[1], results[0], c=results[2], cmap='viridis', s=8, alpha=0.6)
plt.colorbar(sc, label='Sharpe Ratio')
plt.scatter(results[1, max_sharpe_idx], results[0, max_sharpe_idx], marker='*', color='red', s=400, label='Max Sharpe')
plt.scatter(results[1, min_vol_idx], results[0, min_vol_idx], marker='*', color='blue', s=400, label='Min Volatility')
plt.title('Efficient Frontier — CSE Banking Basket')
plt.xlabel('Annualized Volatility'); plt.ylabel('Annualized Return')
plt.legend()
plt.show()

print("Max Sharpe portfolio weights:")
for t, w in zip(TICKERS, weights_record[max_sharpe_idx]):
    print(f"  {t}: {w:.2%}")


**Beginner checkpoint — CSE-specific note:** CSE bank stocks (COMB, HNB, SAMP, NDB, NTB, DFCC) tend to be more correlated with each other than a random 6-stock US basket, since they're all exposed to the same CBSL policy rate cycle and the same macro shocks (like the 2022 default). That shared exposure means the diversification benefit within an all-banking basket will look smaller than a basket spanning different sectors — worth comparing against a mixed-sector basket if you want to see the difference.

---
# Project 5: Value at Risk (VaR) Calculator — CSE Portfolio


In [ ]:
port_weights = weights_record[max_sharpe_idx]
portfolio_returns = returns.dot(port_weights)
portfolio_returns.tail()


In [ ]:
confidence = 0.95

# Historical VaR
historical_var = np.percentile(portfolio_returns, (1 - confidence) * 100)

# Parametric VaR
mu = portfolio_returns.mean()
sigma_p = portfolio_returns.std()
z = norm.ppf(1 - confidence)
parametric_var = mu + z * sigma_p

# Monte Carlo VaR
np.random.seed(1)
simulated_returns = np.random.normal(mu, sigma_p, 100000)
mc_var = np.percentile(simulated_returns, (1 - confidence) * 100)

print(f"1-day Historical VaR  ({confidence:.0%}): {historical_var:.4%}")
print(f"1-day Parametric VaR  ({confidence:.0%}): {parametric_var:.4%}")
print(f"1-day Monte Carlo VaR ({confidence:.0%}): {mc_var:.4%}")


In [ ]:
breach_returns = portfolio_returns[portfolio_returns <= historical_var]
cvar = breach_returns.mean()
print(f"Historical VaR: {historical_var:.4%}")
print(f"CVaR (Expected Shortfall): {cvar:.4%}")


**Beginner checkpoint — CSE-specific note:** Given your dissertation's finding of genuine regime instability around the 2022 sovereign default, Historical VaR computed over a window that includes 2022 will look very different from one computed only on 2023–2024 data. Try re-running Steps 5.1–5.4 on a pre-2022 vs. post-2022 slice of your data (`portfolio_returns['2023':]` for example) and compare — this is a direct, hands-on way to see the regime-instability effect your thesis already documents statistically.

---
## Wrap-up — CSE Edition

Same 5-project pipeline as the US version, but now built entirely on real CSE data you sourced yourself. Two things worth doing next given your existing dissertation work:

1. **Reuse your existing 14,822-row panel** instead of re-downloading from cse.lk — if you already have COMB/HNB/SAMP/NDB/NTB/DFCC price history from your dissertation dataset, that's a cleaner, already-validated data source than a fresh CSE export.
2. **Extend Project 2** using your dissertation's ARIMAX/CatBoost models instead of a simple SMA crossover as the signal — swapping the signal source is the only change needed; the entire backtesting/Sharpe framework here plugs in directly.
